# SQL Interview Self-Test #2 — Targeted Remediation

**Calibration:** Medium. This is a *focused retest* of the 5 concepts you struggled with in Test #1. No easy wins here — every question hits a specific gap.

**Target concepts (and the Test-#1 origin of each gap):**
1. **Conditional aggregation** — `SUM(CASE WHEN … THEN 1 ELSE 0 END)`. (Test #1 Q8: `COUNT(… ELSE 0)` counted every row.)
2. **Aggregate vs window** — knowing WHEN a window function is the right tool. (Test #1 Q9: used window where GROUP BY was right.)
3. **Gaps-and-islands** — consecutive-day streaks, with a same-day duplicate to force DISTINCT dates. (Test #1 Q5: never solved.)
4. **LAG + filter discipline** — `LAG()` partitioning AND reading the filter requirement. (Test #1 Q10: missed `WHERE status='completed'`.)
5. **Top-N per group with ties** — DENSE_RANK vs ROW_NUMBER on a real tie. (Test #1 Q3: fixed on retry, but the tie case wasn't exercised.)

**Format:** markdown question → empty `show("""…""")` answer cell. Run, self-verify, then ask Claude to grade (chat only — SQL drills don't get archived).

**Note:** the data is the same shape as Test #1 but with two deliberate additions — a salary tie in Engineering and a same-day duplicate order. They matter for NQ3 and NQ5.

**Prereqs:** `pip install duckdb pandas`.

## Setup

Run once. Same 4 tables as Test #1 (`departments`, `employees`, `customers`, `orders`) with two added rows (Deepak; a second cust-1 order on 2025-01-15).

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.execute("""
    CREATE TABLE departments (
        dept_id     INTEGER PRIMARY KEY,
        dept_name   VARCHAR
    );
    INSERT INTO departments VALUES
        (1, 'Engineering'),
        (2, 'Sales'),
        (3, 'Marketing'),
        (4, 'HR'),
        (5, 'Finance');

    CREATE TABLE employees (
        emp_id      INTEGER PRIMARY KEY,
        name        VARCHAR,
        salary      INTEGER,
        dept_id     INTEGER,
        manager_id  INTEGER,
        hire_date   DATE
    );
    INSERT INTO employees VALUES
        (1,  'Anjali',  120000, 1, NULL, '2022-01-15'),
        (2,  'Rohit',    95000, 1, 1,    '2022-03-20'),
        (3,  'Priya',   105000, 1, 1,    '2022-06-10'),
        (4,  'Vikram',  130000, 1, 1,    '2023-01-05'),
        (5,  'Suresh',   85000, 2, 6,    '2022-02-15'),
        (6,  'Meera',    90000, 2, NULL, '2021-11-01'),
        (7,  'Karan',    95000, 2, 6,    '2023-03-15'),
        (8,  'Neha',     70000, 3, NULL, '2022-08-20'),
        (9,  'Arjun',    65000, 3, 8,    '2023-05-10'),
        (10, 'Kavita',   75000, 5, NULL, '2023-09-01'),
        (11, 'Deepak',  105000, 1, 1,    '2023-02-01');  -- ties Priya at 105000 in Engineering

    CREATE TABLE customers (
        cust_id      INTEGER PRIMARY KEY,
        name         VARCHAR,
        email        VARCHAR,
        signup_date  DATE
    );
    INSERT INTO customers VALUES
        (1, 'Aman',    'aman@gmail.com',    '2025-01-10'),
        (2, 'Rhea',    'rhea@gmail.com',    '2025-01-12'),
        (3, 'Sahil',   'sahil@yahoo.com',   '2025-02-05'),
        (4, 'Pooja',   'pooja@hotmail.com', '2025-02-20'),
        (5, 'Ravi',    'aman@gmail.com',    '2025-03-01'),
        (6, 'Tanvi',   'tanvi@gmail.com',   '2025-03-15'),
        (7, 'Nikhil',  'sahil@yahoo.com',   '2025-04-02'),
        (8, 'Aisha',   'aisha@gmail.com',   '2025-04-20');

    CREATE TABLE orders (
        order_id    INTEGER PRIMARY KEY,
        cust_id     INTEGER,
        order_date  DATE,
        amount      DECIMAL(10,2),
        status      VARCHAR
    );
    INSERT INTO orders VALUES
        (1,  1, '2025-01-15',  5000.00, 'completed'),
        (2,  1, '2025-01-16',  3000.00, 'completed'),
        (3,  1, '2025-01-17',  2000.00, 'cancelled'),
        (4,  2, '2025-01-20',  4500.00, 'completed'),
        (5,  3, '2025-02-01',  6000.00, 'completed'),
        (6,  3, '2025-02-02',  1500.00, 'pending'),
        (7,  4, '2025-02-10',  8000.00, 'completed'),
        (8,  4, '2025-02-11',  2500.00, 'completed'),
        (9,  1, '2025-02-15',  3500.00, 'completed'),
        (10, 5, '2025-03-05', 10000.00, 'completed'),
        (11, 5, '2025-03-06',  4000.00, 'cancelled'),
        (12, 5, '2025-03-07',  2500.00, 'completed'),
        (13, 5, '2025-03-08',  1500.00, 'completed'),
        (14, 6, '2025-03-10',  7000.00, 'pending'),
        (15, 2, '2025-03-15',  5500.00, 'completed'),
        (16, 7, '2025-04-01',  9000.00, 'completed'),
        (17, 7, '2025-04-05',  3000.00, 'cancelled'),
        (18, 8, '2025-04-10', 12000.00, 'completed'),
        (19, 3, '2025-04-15',  4000.00, 'completed'),
        (20, 1, '2025-04-20',  2000.00, 'completed'),
        (21, 4, '2025-05-01',  6500.00, 'completed'),
        (22, 6, '2025-05-03',  8500.00, 'completed'),
        (23, 6, '2025-05-04',  1000.00, 'pending'),
        (24, 2, '2025-05-10',  5000.00, 'completed'),
        (25, 8, '2025-05-15',  7500.00, 'cancelled'),
        (26, 1, '2025-01-15',  1000.00, 'completed');  -- 2nd order same day as order 1 (cust 1)
""")

def show(sql):
    """Run a SQL query and return the result as a DataFrame."""
    return con.execute(sql).df()

print('Setup complete. 11 employees, 26 orders loaded.')

Setup complete. 11 employees, 26 orders loaded.


## Sample data preview

Note the two new rows: Deepak (Engineering, 105000) and order 26 (cust 1, same day as order 1).

In [3]:
show('SELECT * FROM employees ORDER BY dept_id, salary DESC')

,emp_id,name,salary,dept_id,manager_id,hire_date
0,4,Vikram,130000,1,1,2023-01-05
1,1,Anjali,120000,1,<NA>,2022-01-15
2,3,Priya,105000,1,1,2022-06-10
3,11,Deepak,105000,1,1,2023-02-01
4,2,Rohit,95000,1,1,2022-03-20
5,7,Karan,95000,2,6,2023-03-15
6,6,Meera,90000,2,<NA>,2021-11-01
7,5,Suresh,85000,2,6,2022-02-15
8,8,Neha,70000,3,<NA>,2022-08-20
9,9,Arjun,65000,3,8,2023-05-10


In [4]:
show('SELECT * FROM departments')

,dept_id,dept_name
0,1,Engineering
1,2,Sales
2,3,Marketing
3,4,HR
4,5,Finance


In [5]:
show('SELECT * FROM customers')

,cust_id,name,email,signup_date
0,1,Aman,aman@gmail.com,2025-01-10
1,2,Rhea,rhea@gmail.com,2025-01-12
2,3,Sahil,sahil@yahoo.com,2025-02-05
3,4,Pooja,pooja@hotmail.com,2025-02-20
4,5,Ravi,aman@gmail.com,2025-03-01
5,6,Tanvi,tanvi@gmail.com,2025-03-15
6,7,Nikhil,sahil@yahoo.com,2025-04-02
7,8,Aisha,aisha@gmail.com,2025-04-20


In [6]:
show('SELECT * FROM orders WHERE cust_id = 1 ORDER BY order_date')

,order_id,cust_id,order_date,amount,status
0,1,1,2025-01-15,5000.0,completed
1,26,1,2025-01-15,1000.0,completed
2,2,1,2025-01-16,3000.0,completed
3,3,1,2025-01-17,2000.0,cancelled
4,9,1,2025-02-15,3500.0,completed
5,20,1,2025-04-20,2000.0,completed


---
## NQ1 — Salary-band breakdown per department  *(conditional aggregation)*

For **each department that has employees**, return one row with:
- `dept_name`
- `total_employees`
- `high_earners` — count of employees earning **≥ 100000**
- `low_earners` — count earning **< 100000**
- `pct_high` — percentage of the department earning ≥ 100000, rounded to **1 decimal place**

Order by `pct_high` descending.

**Tables:** `employees`, `departments`.

*This is the `SUM(CASE WHEN …)` pattern. Remember why `COUNT(CASE WHEN … ELSE 0 END)` breaks.*

In [7]:
# NQ1

show("""
SELECT dept_name,
    COUNT(DISTINCT emp_id) AS total_employees,
    COALESCE(SUM(CASE WHEN salary>=100000 THEN 1 END), 0) AS high_earners,
    COALESCE(SUM(CASE WHEN salary<100000 THEN 1 END), 0) AS low_earners,
    ROUND((high_earners/total_employees)*100, 1) AS pct_high
FROM (SELECT  d.dept_name AS dept_name, e.salary AS salary, e.emp_id AS emp_id
     FROM employees e
     LEFT JOIN departments d
     ON e.dept_id=d.dept_id)
GROUP BY dept_name
ORDER BY pct_high
""")

,dept_name,total_employees,high_earners,low_earners,pct_high
0,Sales,3,0.0,3.0,0.0
1,Marketing,2,0.0,2.0,0.0
2,Finance,1,0.0,1.0,0.0
3,Engineering,5,4.0,1.0,80.0


---
## NQ2 — Each employee vs their department average  *(window function — the right use)*

Return **one row per employee** with:
- `dept_name`
- `emp_name`
- `salary`
- `dept_avg_salary` — the average salary of that employee's department, rounded to 2 decimals
- `diff_from_avg` — `salary - dept_avg_salary`, rounded to 2 decimals

Order by `dept_name`, then `salary` descending.

**Tables:** `employees`, `departments`.

*Test #1 Q9 wanted one row per department (GROUP BY). This one wants every employee row to carry its department's average alongside it. Which tool is right here, and why is it the opposite choice from Q9?*

In [8]:
# NQ2

show("""
SELECT dept_name, emp_name, salary, dept_avg_salary, salary - dept_avg_salary AS diff_from_avg
FROM (SELECT  d.dept_name AS dept_name, e.salary AS salary, e.emp_id AS emp_id, e.name AS emp_name, AVG(e.salary)OVER(PARTITION BY d.dept_name) AS dept_avg_salary
     FROM employees e
     LEFT JOIN departments d
     ON e.dept_id=d.dept_id)
ORDER BY dept_name, salary DESC
""")

,dept_name,emp_name,salary,dept_avg_salary,diff_from_avg
0,Engineering,Vikram,130000,111000.0,19000.0
1,Engineering,Anjali,120000,111000.0,9000.0
2,Engineering,Priya,105000,111000.0,-6000.0
3,Engineering,Deepak,105000,111000.0,-6000.0
4,Engineering,Rohit,95000,111000.0,-16000.0
5,Finance,Kavita,75000,75000.0,0.0
6,Marketing,Neha,70000,67500.0,2500.0
7,Marketing,Arjun,65000,67500.0,-2500.0
8,Sales,Karan,95000,90000.0,5000.0
9,Sales,Meera,90000,90000.0,0.0


---
## NQ3 — Customers with 3+ consecutive order days  *(gaps-and-islands)*

Find every customer who placed orders on **three or more consecutive calendar days**.

- All statuses count.
- **A customer can have multiple orders on the same day** (see cust 1 on 2025-01-15) — count *distinct days*, not orders.
- Return: `cust_id`. Order by `cust_id`.

**Tables:** `orders`.

*If you skip the distinct-dates step, the duplicate same-day order will silently break your streak logic. Build the streak on distinct dates first.*

In [18]:
show("""
WITH distinct_orders AS (
    SELECT DISTINCT cust_id, order_date FROM orders
    ),
streaks AS (
     SELECT *, CASE WHEN order_date - LAG(order_date, 1)OVER(PARTITION BY cust_id ORDER BY order_date) AS streak
     FROM distinct_orders
    )
SELECT cust_id  FROM streaks GROUP BY cust_id HAVING SUM(streak)>=3 ORDER BY cust_id
""")

ParserException: Parser Error: syntax error at or near "AS"

LINE 6: ...(order_date, 1)OVER(PARTITION BY cust_id ORDER BY order_date) AS streak
                                                                         ^

---
## NQ4 — Order-over-order change per customer  *(LAG + filter discipline)*

**For completed orders only**, return for each customer:
- `order_id`
- `cust_id`
- `order_date`
- `amount`
- `prev_amount` — that customer's previous completed-order amount (NULL for their first)
- `delta` — `amount - prev_amount` (NULL for their first)

Order by `cust_id`, `order_date`, `order_id`.

**Tables:** `orders`.

*Two things to get right: (1) read the filter — cancelled/pending orders must not appear AND must not count as the 'previous' order (the Q10 miss). (2) Customer 1 has two completed orders on the same day (2025-01-15) — `ORDER BY order_date` alone is ambiguous between them, so the window's ORDER BY needs `order_id` as a tiebreaker or your `prev_amount` is non-deterministic.*

In [ ]:
# NQ4

show("""
WITH prev AS (
SELECT *, LAG(amount, 1)OVER(PARTITION BY cust_id ORDER BY order_date) AS prev_amount
FROM orders
WHERE status='completed'
    )
SELECT order_id, cust_id, order_date, amount, prev_amount, amount-prev_amount AS delta
FROM prev
ORDER BY cust_id, order_date, order_id
""")

,order_id,cust_id,order_date,amount,prev_amount,delta
0,1,1,2025-01-15,5000.0,NaN,NaN
1,26,1,2025-01-15,1000.0,5000.0,-4000.0
2,2,1,2025-01-16,3000.0,1000.0,2000.0
3,9,1,2025-02-15,3500.0,3000.0,500.0
4,20,1,2025-04-20,2000.0,3500.0,-1500.0
5,4,2,2025-01-20,4500.0,NaN,NaN
6,15,2,2025-03-15,5500.0,4500.0,1000.0
7,24,2,2025-05-10,5000.0,5500.0,-500.0
8,5,3,2025-02-01,6000.0,NaN,NaN
9,19,3,2025-04-15,4000.0,6000.0,-2000.0


---
## NQ5 — Top 2 salary tiers per department  *(DENSE_RANK + ties)*

For each department, return employees in the **top 2 salary tiers**, where a *tier* is a distinct salary level (ties share a tier).

- Use `DENSE_RANK` so two employees on the same salary share the same rank, and the next distinct salary is the following rank.
- Return: `dept_name`, `emp_name`, `salary`, `salary_tier`.
- Order by `dept_name`, then `salary_tier`, then `emp_name`.

**Tables:** `employees`, `departments`.

*Engineering now has a tie: Priya and Deepak both earn 105000. Note how many Engineering rows your query returns. Then, in a comment, state how the result would differ if you used `ROW_NUMBER()` instead of `DENSE_RANK()` — that difference is the whole point of this question.*

In [ ]:
# NQ5

show("""
WITH joined AS (
SELECT d.dept_name AS dept_name, e.name AS emp_name, e.salary AS salary
FROM employees e
JOIN departments d
ON e.dept_id=d.dept_id
    ),
tiers AS (
SELECT *, DENSE_RANK() OVER(PARTITION BY dept_name ORDER BY salary DESC) AS salary_tier
FROM joined
    )
SELECT * FROM tiers WHERE salary_tier<=2 ORDER BY dept_name, salary_tier, emp_name;
""")

# Then answer in a comment: how would ROW_NUMBER() change the Engineering rows?

,dept_name,emp_name,salary,salary_tier
0,Engineering,Vikram,130000,1
1,Engineering,Anjali,120000,2
2,Finance,Kavita,75000,1
3,Marketing,Neha,70000,1
4,Marketing,Arjun,65000,2
5,Sales,Karan,95000,1
6,Sales,Meera,90000,2


---
## Self-verify checklist (tuned to your 5 gaps)

- **NQ1:** Did you use `SUM(CASE WHEN … THEN 1 ELSE 0 END)` — NOT `COUNT(… ELSE 0)`? Does `high + low = total` per row? Is `pct_high` a real percentage (×100), rounded to 1 dp?
- **NQ2:** Did you use a **window** function (one row per employee), not GROUP BY (one row per dept)? Row count should equal the employee count (11), not the department count.
- **NQ3:** Did you `SELECT DISTINCT cust_id, order_date` before the streak logic? Does cust 1 still qualify despite the same-day duplicate?
- **NQ4:** Did you filter `WHERE status = 'completed'` BEFORE the LAG, so the previous-order lookup skips cancelled/pending? Is `delta` NULL for each customer's first completed order?
- **NQ5:** How many Engineering rows? (Tier 1 = Vikram; tier 2 = Anjali — or does the tie change what tier 2 means?) Did you articulate the ROW_NUMBER difference?

When done, paste your 5 answers in chat for grading.